In [ ]:
!pip install transformers torch accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.9 MB/s eta 0:00:00


In [ ]:
import sqlite3

# Create a database
conn = sqlite3.connect("travel.db")
cursor = conn.cursor()

# Create hotels table
cursor.execute("""
CREATE TABLE hotels (
    city TEXT,
    hotel_name TEXT,
    price_per_night INTEGER
)
""")

# Create attractions table
cursor.execute("""
CREATE TABLE attractions (
    city TEXT,
    attraction_name TEXT,
    ticket_price INTEGER
)
""")

conn.commit()

print("Database created successfully!")


Database created successfully!


In [ ]:
# Insert hotel data
hotels_data = [
    ("Paris", "Hotel Lumiere", 220),
    ("Paris", "Budget Paris Inn", 120),
    ("Rome", "Roma Palace", 180),
    ("Rome", "Colosseum Stay", 130),
    ("Tokyo", "Shibuya Hotel", 200),
    ("Tokyo", "Tokyo Budget Lodge", 110)
]

cursor.executemany("INSERT INTO hotels VALUES (?, ?, ?)", hotels_data)

# Insert attraction data
attractions_data = [
    ("Paris", "Eiffel Tower", 30),
    ("Paris", "Louvre Museum", 25),
    ("Rome", "Colosseum", 20),
    ("Rome", "Vatican Museum", 22),
    ("Tokyo", "Tokyo Tower", 15),
    ("Tokyo", "Shinjuku Gyoen", 10)
]

cursor.executemany("INSERT INTO attractions VALUES (?, ?, ?)", attractions_data)

conn.commit()

print("Sample travel data inserted!")


Sample travel data inserted!


In [ ]:
def search_hotels(city, max_price=None):
    if max_price:
        cursor.execute(
            "SELECT * FROM hotels WHERE city=? AND price_per_night<=?",
            (city, max_price)
        )
    else:
        cursor.execute(
            "SELECT * FROM hotels WHERE city=?",
            (city,)
        )

    return cursor.fetchall()


In [ ]:
print(search_hotels("Paris"))

[('Paris', 'Hotel Lumiere', 220), ('Paris', 'Budget Paris Inn', 120)]


In [ ]:
def search_attractions(city):
    cursor.execute(
        "SELECT * FROM attractions WHERE city=?",
        (city,)
    )

    return cursor.fetchall()


In [ ]:
print(search_attractions("Rome"))


[('Rome', 'Colosseum', 20), ('Rome', 'Vatican Museum', 22)]


In [ ]:
def simple_assistant(query):
    query = query.lower()

    if "hotel" in query:
        if "paris" in query:
            return search_hotels("Paris")
        if "rome" in query:
            return search_hotels("Rome")
        if "tokyo" in query:
            return search_hotels("Tokyo")

    if "attraction" in query:
        if "paris" in query:
            return search_attractions("Paris")
        if "rome" in query:
            return search_attractions("Rome")
        if "tokyo" in query:
            return search_attractions("Tokyo")

    return "Sorry, I couldn't understand."


In [ ]:
print(simple_assistant("Show me hotels in Rome"))


[('Rome', 'Roma Palace', 180), ('Rome', 'Colosseum Stay', 130)]


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "microsoft/phi-3-mini-4k-instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("Model loaded successfully!")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

Model loaded successfully!


In [ ]:
def generate_response(prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            temperature=0.7
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)


In [ ]:
print(generate_response("Suggest a 3-day trip to Rome."))


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Suggest a 3-day trip to Rome. Include a mix of historical sites, local cuisine, and leisure activities. Ensure the itinerary is suitable for a family with young children. Day 1:


- Morning: Visit the Colosseum and Roman Forum.

- Afternoon: Lunch at Armando al Pantheon, known for its traditional Roman dishes.

- Evening: Stroll through Piazza Navona and enjoy gelato at Gelateria del Teatro.


Day 2:


- Morning: Explore the Vatican Museums and Sistine Chapel.

- Afternoon: Picnic in the Gardens of Vat


In [ ]:
def travel_assistant(query):
    query_lower = query.lower()

    # Tool usage
    if "hotel" in query_lower:
        if "paris" in query_lower:
            results = search_hotels("Paris")
        elif "rome" in query_lower:
            results = search_hotels("Rome")
        elif "tokyo" in query_lower:
            results = search_hotels("Tokyo")
        else:
            results = []

        prompt = f"""
        The user asked: {query}

        Here are the available hotels from the database:
        {results}

        Provide a helpful recommendation using this data.
        """

        return generate_response(prompt)

    # If no tool needed
    return generate_response(query)


In [ ]:
print(travel_assistant("Show me hotels in Rome under $200"))



        The user asked: Show me hotels in Rome under $200

        Here are the available hotels from the database:
        [('Rome', 'Roma Palace', 180), ('Rome', 'Colosseum Stay', 130)]

        Provide a helpful recommendation using this data.
        
        Answer:

        Based on your budget of under $200 for a hotel in Rome, I recommend considering the 'Colosseum Stay' hotel. It's priced at $130, which is well within your budget and offers a unique experience by being located near the iconic Colosseum. This could provide you with a memorable stay in the heart of Rome.





In [1]:
!git clone https://github.com/lavanya-4/Travel_Trip_Advisor.git

Cloning into 'Travel_Trip_Advisor'...
remote: Enumerating objects: 3, done.
remote: Counting objects: 100% (3/3), done.
remote: Total 3 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (3/3), done.


In [2]:
%cd Travel_Trip_Advisor

/content/Travel_Trip_Advisor


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
!ls /content

drive  sample_data  Travel_Trip_Advisor


In [6]:
%cd /content/Travel_Trip_Advisor

/content/Travel_Trip_Advisor


In [7]:
!ls "/content/drive/My Drive/Colab Notebooks"

 017553718.ipynb
'Activity-data-pre-processing (1).ipynb'
 AI_2021_trail.ipynb
 Classification.ipynb
'Copy of AI_paper_implementation.ipynb'
'Copy of cleaned_code.ipynb'
 disease_symptom_dataset.csv
 DL_HW.ipynb
 Document_Clustering_Assignment.ipynb
 Embedding_Techniques.ipynb
 fairness_income_final_optimized.ipynb
 fairness_income_modified.ipynb
 fairness_income_threshold_tuned.ipynb
 final_cf_recommender_colab.ipynb
 hw2_017553718.ipynb
 Hw2_DL.ipynb
 hybrid_blended_model_colab_safe.ipynb
 hybrid_clean_preprocessing_rmse.ipynb
 lightgcn_deep_dive.ipynb
 lightgcn.yaml
 Matrix_operations_017553718
 music_recommender_sparse_svd.ipynb
'NeuralCF (1).ipynb'
 nGramsLanguageModels_ClassActivity.ipynb
'practiceC!.ipynb'
 Prediction_of_disease_symptoms_based_on_decision_tree.ipynb
 rs.ipynb
 RS.ipynb
'Sentiment Classification Using Pretrained and Fine-Tuned BERT-tiny Models.ipynb'
 shared.ipynb
 simple_transformer_example.ipynb
 Spring2026_TextExtraction_web_ppt_pdf.ipynb
 Travel_Trip.ipynb
 U